# Эксперимент TabBERT с конкатенацией числового признака

Этот ноутбук соответствует варианту, в котором `transaction_bert` и `history_bert` кодируют только категориальные представления, а числовой признак добавляется к выходному представлению непосредственно перед классификационной головой.

Этот файл полезно читать как отдельную абляцию по сравнению с базовой моделью с числовой веткой: здесь проверяется, достаточно ли поздней конкатенации числовой информации без полного включения числа в иерархическую архитектуру.

In [1]:
import random
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertConfig, BertModel

2026-05-03 22:59:23.659702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777838363.687187  248003 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777838363.695197  248003 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777838363.716544  248003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777838363.716586  248003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777838363.716589  248003 computation_placer.cc:177] computation placer alr

In [2]:
class TabBertVocabBuilder:
    """Класс для создания словаря признаков транзакций и границ бинов."""

    def __init__(self, df, categorical_columns, numerical_columns, special_tokens=None):
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns

        if special_tokens is None:
            self.special_tokens = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]']
        else:
            self.special_tokens = special_tokens

        self.bin_edges = {}
        self.vocab = self._build_vocab(df)

    def _build_vocab(self, df):
        """Создание словаря из уникальных значений категориальных колонок."""
        vocab_dict = {}

        for i, token in enumerate(self.special_tokens):
            vocab_dict[token] = i

        current_idx = len(self.special_tokens)

        for col in self.categorical_columns:
            unique_values = df[col].astype(str).unique()
            for val in unique_values:
                token = f"{col}_{val}"
                if token not in vocab_dict:
                    vocab_dict[token] = current_idx
                    current_idx += 1

        return vocab_dict

    def save(self, path):
        """Сохранить словарь и метаданные."""
        save_dict = {
            'vocab': self.vocab,
            'bin_edges': self.bin_edges,
            'categorical_columns': self.categorical_columns,
            'numerical_columns': self.numerical_columns,
            'special_tokens': self.special_tokens,
        }

        with open(path, 'wb') as f:
            pickle.dump(save_dict, f)
        print(f"TabBertVocab saved to {path}")

    @classmethod
    def load(cls, path):
        """Загрузить словарь и метаданные."""
        with open(path, 'rb') as f:
            save_dict = pickle.load(f)

        instance = cls.__new__(cls)
        instance.vocab = save_dict['vocab']
        instance.bin_edges = save_dict['bin_edges']
        instance.categorical_columns = save_dict['categorical_columns']
        instance.numerical_columns = save_dict['numerical_columns']
        instance.special_tokens = save_dict['special_tokens']

        print(f"TabBertVocab loaded from {path}. Size: {len(instance.vocab)}")
        return instance

In [3]:
class TabBertTokenizer:
    """Токенизатор для табличных данных."""

    def __init__(self, vocab_builder, max_length=128):
        self.vocab_builder = vocab_builder
        self.vocab = vocab_builder.vocab
        self.max_length = max_length
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def tokenize_transaction(self, transaction_row, categorical_columns=None, numerical_columns=None):
        """
        Токенизация одной транзакции.

        Returns:
            tuple: (categorical_ids, numerical_values, tokens)
        """
        if categorical_columns is None:
            categorical_columns = self.vocab_builder.categorical_columns
        if numerical_columns is None:
            numerical_columns = self.vocab_builder.numerical_columns

        tokens = ['[CLS]']
        categorical_ids = [self.vocab.get('[CLS]')]
        numerical_values = []

        for col in categorical_columns:
            if col in transaction_row.index:
                val = str(transaction_row[col]) if not pd.isna(transaction_row[col]) else 'missing'
                token = f"{col}_{val}"
                token_id = self.vocab.get(token, self.vocab['[UNK]'])
                tokens.append(token)
                categorical_ids.append(token_id)
            else:
                tokens.append('[UNK]')
                categorical_ids.append(self.vocab['[UNK]'])

        for col in numerical_columns:
            if col in transaction_row.index and not pd.isna(transaction_row[col]):
                val = float(transaction_row[col])
                numerical_values.append(val)
                tokens.append(f"{col}_{val:.2f}")
            else:
                numerical_values.append(0.0)
                tokens.append(f"{col}_missing")

        tokens.append('[SEP]')
        categorical_ids.append(self.vocab.get('[SEP]'))

        if len(categorical_ids) > self.max_length:
            categorical_ids = categorical_ids[:self.max_length]
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        current_len = len(categorical_ids)
        if current_len < self.max_length:
            padding_len = self.max_length - current_len
            categorical_ids += [self.vocab['[PAD]']] * padding_len
            tokens += ['[PAD]'] * padding_len

        return categorical_ids, numerical_values, tokens

    def convert_tokens_to_ids(self, tokens):
        """Конвертация токенов в индексы."""
        return [self.vocab.get(token, self.vocab['[UNK]']) for token in tokens]

    def convert_ids_to_tokens(self, ids):
        """Конвертация индексов в токены."""
        return [self.inv_vocab.get(id_one, '[UNK]') for id_one in ids]

In [ ]:
import pandas as pd

df = pd.read_csv('card_transaction.v1.prepare.csv')

In [5]:
df.head(2)

,user,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,is_fraud,hour,minute,timestamp,isrefund
0,0,0,2002,9,1,06:21,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0,6,21,1030861260000000000,1
1,0,0,2002,9,1,06:42,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,6,42,1030862520000000000,1


In [6]:
# categorical_columns = ['user_id', 'Card', 'Year', 'Month' , 'Day', 'hour', 'minute',  'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'is_refund']
categorical_columns = ['user', 'Card',  'Month' , 'Day', 'hour', 'minute',  'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'isrefund']
numerical_columns = ['Amount']


# vocab_builder = TabBertVocabBuilder(df, categorical_columns, numerical_columns)
# vocab_builder.save('tabbert_v7/tabbert_vocab_all.pkl')



vocab_builder = TabBertVocabBuilder.load('tabbert_v7/tabbert_vocab_all.pkl')

tokenizer = TabBertTokenizer(vocab_builder, max_length=32)





TabBertVocab loaded from tabbert_v7/tabbert_vocab_all.pkl. Size: 143597


In [7]:
class TabBertMLMDataset(Dataset):
    """Dataset для MLM pretraining: current transaction + history window."""

    def __init__(self, df, tokenizer, categorical_columns, numerical_columns,
                 max_seq_length=128, history_size=10, stride=None, mask_prob=0.15):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.max_seq_length = max_seq_length
        self.history_size = history_size
        self.stride = stride if stride is not None else history_size
        self.mask_prob = mask_prob

        self.cls_id = self.tokenizer.vocab['[CLS]']
        self.sep_id = self.tokenizer.vocab['[SEP]']
        self.pad_id = self.tokenizer.vocab['[PAD]']
        self.unk_id = self.tokenizer.vocab['[UNK]']
        self.mask_id = self.tokenizer.vocab['[MASK]']
        self.vocab_size = len(self.tokenizer.vocab)
        self.num_numerical_features = len(self.numerical_columns)
        self.special_ids = {self.cls_id, self.sep_id, self.pad_id}

        used_columns = list(dict.fromkeys(self.categorical_columns + self.numerical_columns))
        self.column_arrays = {
            col: self.df[col].to_numpy(copy=False) for col in used_columns
        }

        self.pad_cat_ids = np.full(self.max_seq_length, self.pad_id, dtype=np.int64)
        self.pad_num_values = np.zeros(self.num_numerical_features, dtype=np.float32)
        self.pad_mask = np.zeros(self.max_seq_length, dtype=np.int64)

        self.cat_value_to_id = {col: {} for col in self.categorical_columns}
        for token, token_id in self.tokenizer.vocab.items():
            if token.startswith('['):
                continue
            col_name, sep, value = token.partition('_')
            if sep and col_name in self.cat_value_to_id:
                self.cat_value_to_id[col_name][value] = token_id

        history_windows = []
        current_indices = []
        client_transactions = self.df.groupby('user').indices

        for indices in client_transactions.values():
            indices = sorted(indices)
            if not indices:
                continue

            if len(indices) <= history_size:
                history_indices = indices[:-1]
                padded_history = [-1] * (history_size - len(history_indices)) + history_indices
                history_windows.append(padded_history)
                current_indices.append(indices[-1])
            else:
                for i in range(history_size, len(indices), self.stride):
                    history_windows.append(indices[i-history_size:i])
                    current_indices.append(indices[i])

        self.history_indices = np.asarray(history_windows, dtype=np.int64)
        self.current_indices = np.asarray(current_indices, dtype=np.int64)

        print(f"Создано {len(self.current_indices)} окон из {len(df)} транзакций")

    def __len__(self):
        return len(self.current_indices)

    def _tokenize_transaction(self, idx):
        """Быстрая токенизация без pandas.iloc внутри training loop."""
        if idx is None or idx < 0:
            return self.pad_cat_ids.copy(), self.pad_num_values.copy(), self.pad_mask.copy()

        cat_ids = self.pad_cat_ids.copy()
        num_values = self.pad_num_values.copy()
        mask = self.pad_mask.copy()

        write_pos = 0
        cat_ids[write_pos] = self.cls_id
        write_pos += 1

        for col in self.categorical_columns:
            if write_pos >= self.max_seq_length - 1:
                break

            value = self.column_arrays[col][idx]
            if pd.isna(value):
                token_id = self.unk_id
            else:
                token_id = self.cat_value_to_id[col].get(str(value), self.unk_id)

            cat_ids[write_pos] = token_id
            write_pos += 1

        if write_pos < self.max_seq_length:
            cat_ids[write_pos] = self.sep_id
            write_pos += 1

        mask[:write_pos] = 1

        for num_idx, col in enumerate(self.numerical_columns):
            value = self.column_arrays[col][idx]
            num_values[num_idx] = 0.0 if pd.isna(value) else float(value)

        return cat_ids, num_values, mask

    def _apply_masking(self, cat_ids, num_values):
        """Размечаем labels только для реально выбранных categorical MLM-позиций."""
        masked_cat_ids = cat_ids.copy()
        cat_labels = np.full_like(cat_ids, fill_value=-100)
        masked_num_values = num_values.copy()
        num_labels = num_values.copy().astype(np.float32)

        for i, token_id in enumerate(cat_ids):
            if token_id in self.special_ids or token_id == self.unk_id:
                continue

            if random.random() < self.mask_prob:
                cat_labels[i] = token_id

                r = random.random()
                if r < 0.8:
                    masked_cat_ids[i] = self.mask_id
                elif r < 0.9:
                    masked_cat_ids[i] = random.randint(0, self.vocab_size - 1)

        return masked_cat_ids, cat_labels, masked_num_values, num_labels

    def __getitem__(self, idx):
        current_idx = int(self.current_indices[idx])
        history_indices = self.history_indices[idx]

        history_cat_ids = np.empty((self.history_size, self.max_seq_length), dtype=np.int64)
        history_num_values = np.empty((self.history_size, self.num_numerical_features), dtype=np.float32)
        history_masks = np.empty((self.history_size, self.max_seq_length), dtype=np.int64)
        history_cat_labels = np.empty((self.history_size, self.max_seq_length), dtype=np.int64)
        history_num_labels = np.empty((self.history_size, self.num_numerical_features), dtype=np.float32)

        for pos, hist_idx in enumerate(history_indices):
            cat_ids, num_values, mask = self._tokenize_transaction(int(hist_idx))
            masked_cat_ids, cat_labels, masked_num_values, num_labels = self._apply_masking(cat_ids, num_values)

            history_cat_ids[pos] = masked_cat_ids
            history_num_values[pos] = masked_num_values
            history_masks[pos] = mask
            history_cat_labels[pos] = cat_labels
            history_num_labels[pos] = num_labels

        current_cat_ids, current_num_values, current_mask = self._tokenize_transaction(current_idx)
        masked_current_cat_ids, current_cat_labels, masked_current_num_values, current_num_labels = \
            self._apply_masking(current_cat_ids, current_num_values)

        return {
            'current_cat_ids': torch.from_numpy(masked_current_cat_ids.copy()).long(),
            'current_num_values': torch.from_numpy(masked_current_num_values.copy()).float(),
            'current_mask': torch.from_numpy(current_mask.copy()).long(),
            'current_cat_labels': torch.from_numpy(current_cat_labels.copy()).long(),
            'current_num_labels': torch.from_numpy(current_num_labels.copy()).float(),
            'history_cat_ids': torch.from_numpy(history_cat_ids.copy()).long(),
            'history_num_values': torch.from_numpy(history_num_values.copy()).float(),
            'history_mask': torch.from_numpy(history_masks.copy()).long(),
            'history_cat_labels': torch.from_numpy(history_cat_labels.copy()).long(),
            'history_num_labels': torch.from_numpy(history_num_labels.copy()).float(),
        }

In [8]:
class TabBertFraudDataset(Dataset):
    """Dataset для fine-tuning на fraud detection, совместимый с forward(cat_ids, num_values, attention_mask, labels)."""

    def __init__(self, df, tokenizer, categorical_columns, numerical_columns,
                 user_col='user', time_col='timestamp', label_col='is_fraud',
                 max_seq_length=128, window_size=10, stride=None, min_window_size=2):
        self.df = df.sort_values([user_col, time_col]).reset_index(drop=True)
        self.tokenizer = tokenizer
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.user_col = user_col
        self.time_col = time_col
        self.label_col = label_col
        self.max_seq_length = max_seq_length
        self.window_size = window_size
        self.stride = stride if stride is not None else window_size
        self.min_window_size = min_window_size

        self.cls_id = self.tokenizer.vocab['[CLS]']
        self.sep_id = self.tokenizer.vocab['[SEP]']
        self.pad_id = self.tokenizer.vocab['[PAD]']
        self.unk_id = self.tokenizer.vocab['[UNK]']
        self.num_numerical_features = len(self.numerical_columns)

        used_columns = list(dict.fromkeys(
            [self.user_col, self.time_col, self.label_col] + self.categorical_columns + self.numerical_columns
        ))
        self.column_arrays = {
            col: self.df[col].to_numpy(copy=False) for col in used_columns
        }

        self.pad_cat_ids = np.full(self.max_seq_length, self.pad_id, dtype=np.int64)
        self.pad_num_values = np.zeros(self.num_numerical_features, dtype=np.float32)
        self.pad_mask = np.zeros(self.max_seq_length, dtype=np.int64)

        self.cat_value_to_id = {col: {} for col in self.categorical_columns}
        for token, token_id in self.tokenizer.vocab.items():
            if token.startswith('['):
                continue
            col_name, sep, value = token.partition('_')
            if sep and col_name in self.cat_value_to_id:
                self.cat_value_to_id[col_name][value] = token_id

        self.windows = []
        self._create_windows()

        print(f"Создано {len(self.windows)} fraud-окон из {len(self.df)} транзакций")

    def _create_windows(self):
        """Создает окна фиксированного размера для каждого пользователя."""
        grouped_indices = self.df.groupby(self.user_col).indices

        for indices in grouped_indices.values():
            indices = sorted(indices)
            if not indices:
                continue

            for start_idx in range(0, len(indices), self.stride):
                end_idx = min(start_idx + self.window_size, len(indices))
                window_indices = indices[start_idx:end_idx]

                if len(window_indices) >= self.min_window_size:
                    self.windows.append({
                        'indices': window_indices,
                        'actual_size': len(window_indices),
                    })

    def _tokenize_transaction(self, idx):
        """Токенизация одной транзакции в формат, ожидаемый classification forward."""
        if idx is None or idx < 0:
            return self.pad_cat_ids.copy(), self.pad_num_values.copy(), self.pad_mask.copy()

        cat_ids = self.pad_cat_ids.copy()
        num_values = self.pad_num_values.copy()
        mask = self.pad_mask.copy()

        write_pos = 0
        cat_ids[write_pos] = self.cls_id
        write_pos += 1

        for col in self.categorical_columns:
            if write_pos >= self.max_seq_length - 1:
                break

            value = self.column_arrays[col][idx]
            if pd.isna(value):
                token_id = self.unk_id
            else:
                token_id = self.cat_value_to_id[col].get(str(value), self.unk_id)

            cat_ids[write_pos] = token_id
            write_pos += 1

        if write_pos < self.max_seq_length:
            cat_ids[write_pos] = self.sep_id
            write_pos += 1

        mask[:write_pos] = 1

        for num_idx, col in enumerate(self.numerical_columns):
            value = self.column_arrays[col][idx]
            num_values[num_idx] = 0.0 if pd.isna(value) else float(value)

        return cat_ids, num_values, mask

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        window = self.windows[idx]
        window_indices = window['indices']
        actual_size = window['actual_size']

        all_cat_ids = np.empty((self.window_size, self.max_seq_length), dtype=np.int64)
        all_num_values = np.empty((self.window_size, self.num_numerical_features), dtype=np.float32)
        all_masks = np.empty((self.window_size, self.max_seq_length), dtype=np.int64)
        all_labels = np.full((self.window_size,), fill_value=-100, dtype=np.int64)

        for pos in range(self.window_size):
            if pos < actual_size:
                row_idx = int(window_indices[pos])
                cat_ids, num_values, mask = self._tokenize_transaction(row_idx)
                label = int(self.column_arrays[self.label_col][row_idx])
            else:
                cat_ids, num_values, mask = self._tokenize_transaction(-1)
                label = -100

            all_cat_ids[pos] = cat_ids
            all_num_values[pos] = num_values
            all_masks[pos] = mask
            all_labels[pos] = label

        return {
            'cat_ids': torch.from_numpy(all_cat_ids.copy()).long(),
            'num_values': torch.from_numpy(all_num_values.copy()).float(),
            'attention_mask': torch.from_numpy(all_masks.copy()).long(),
            'labels': torch.from_numpy(all_labels.copy()).long(),
            'actual_size': actual_size,
        }

In [9]:
class TabBertForFraudDetectionNumericConcat(nn.Module):
    def __init__(
        self,
        vocab_size,
        field_ranges,
        inv_vocab,
        num_numerical_features=1,
        hidden_size=128,
        num_classes=2,
        max_history=10,
        freeze_first_bert=False,
        num_hidden_layers=1,
    ):
        super().__init__()

        self.field_ranges = field_ranges
        self.inv_vocab = inv_vocab
        self.hidden_size = hidden_size
        self.num_numerical_features = num_numerical_features
        self.field_vector_size = hidden_size
        self.classifier_input_size = hidden_size + num_numerical_features
        self.num_attention_heads = self._choose_num_attention_heads(hidden_size)

        self.cat_embedding = nn.Embedding(vocab_size, hidden_size)
        self.num_norm = nn.BatchNorm1d(num_numerical_features)

        config = BertConfig(
            hidden_size=hidden_size,
            num_hidden_layers=3,
            num_attention_heads=self.num_attention_heads,
            max_position_embeddings=196,
            type_vocab_size=2,
            intermediate_size=hidden_size * 4,
        )
        self.transaction_bert = BertModel(config)

        config_history = BertConfig(
            hidden_size=hidden_size,
            num_hidden_layers=2,
            num_attention_heads=self.num_attention_heads,
            intermediate_size=hidden_size * 2,
            max_position_embeddings=1024,
            hidden_dropout_prob=0.3,
            attention_probs_dropout_prob=0.3,
        )
        self.history_bert = BertModel(config_history)

        self.classifier = nn.Sequential(
            nn.Linear(self.classifier_input_size, self.classifier_input_size),
            nn.BatchNorm1d(self.classifier_input_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(self.classifier_input_size, self.classifier_input_size // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(self.classifier_input_size // 2, num_classes),
        )

        self.mlm_head = nn.Linear(hidden_size, vocab_size)
        # self.num_head = nn.Linear(hidden_size, num_numerical_features)

        self.register_buffer("class_weights", torch.tensor([1.0, 5.0]))

        if freeze_first_bert:
            for param in self.transaction_bert.parameters():
                param.requires_grad = False

    @staticmethod
    def _choose_num_attention_heads(hidden_size):
        for candidate in (8, 4, 3, 2, 1):
            if hidden_size % candidate == 0:
                return candidate
        return 1

    def _normalize_num_values(self, num_values, transaction_mask=None):
        """Нормализует числовые признаки перед классификационной головой."""
        original_shape = num_values.shape
        flat_num_values = num_values.reshape(-1, self.num_numerical_features)

        if transaction_mask is None:
            if self.training and flat_num_values.size(0) <= 1:
                return num_values
            return self.num_norm(flat_num_values).reshape(original_shape)

        flat_mask = transaction_mask.reshape(-1).bool()
        normalized_num_values = flat_num_values.clone()

        if flat_mask.any():
            valid_num_values = flat_num_values[flat_mask]
            if self.training and valid_num_values.size(0) <= 1:
                normalized_num_values[flat_mask] = valid_num_values
            else:
                normalized_num_values[flat_mask] = self.num_norm(valid_num_values)

        return normalized_num_values.reshape(original_shape)

    def encode_transaction(self, cat_ids, num_values, attention_mask):
        """Кодирует транзакцию только по категориальным embeddings."""
        field_embeds = self.cat_embedding(cat_ids)

        outputs = self.transaction_bert(
            inputs_embeds=field_embeds,
            attention_mask=attention_mask,
        )

        return outputs.last_hidden_state

    def _encode_cls_sequence(self, cat_ids, num_values, attention_mask):
        """Представление каждой транзакции в окне: [batch, seq_len, hidden_size]."""
        batch_size, seq_len = cat_ids.shape[:2]
        num_cat_fields = cat_ids.size(-1)

        flat_cat_ids = cat_ids.reshape(batch_size * seq_len, -1)
        flat_num_values = num_values.reshape(batch_size * seq_len, -1)
        flat_attention_mask = attention_mask.reshape(batch_size * seq_len, -1)

        flat_transaction_embeds = self.encode_transaction(
            flat_cat_ids,
            flat_num_values,
            flat_attention_mask,
        )

        transaction_embeds = flat_transaction_embeds.reshape(
            batch_size,
            seq_len * num_cat_fields,
            self.field_vector_size,
        )
        valid_mask = attention_mask.reshape(batch_size, seq_len * num_cat_fields)

        outputs = self.history_bert(
            inputs_embeds=transaction_embeds,
            attention_mask=valid_mask,
        )

        sequence_output = outputs.last_hidden_state.reshape(
            batch_size,
            seq_len,
            num_cat_fields,
            self.field_vector_size,
        )[:, :, 0, :]
        return sequence_output

    def _pool_transaction_fields(self, field_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(dtype=field_embeddings.dtype)
        summed = (field_embeddings * mask).sum(dim=2)
        denom = mask.sum(dim=2).clamp_min(1.0)
        return summed / denom

    def forward_mlm(
        self,
        current_cat_ids,
        current_num_values,
        current_mask,
        history_cat_ids,
        history_num_values,
        history_mask,
        current_cat_labels,
        current_num_labels,
        history_cat_labels,
        history_num_labels,
    ):
        """MLM forward без отдельного numeric-token: BERT видит только категориальные embeddings."""
        batch_size = current_cat_ids.size(0)
        total_transactions = 1 + history_cat_ids.size(1)
        num_cat_fields = current_cat_ids.size(1)

        all_cat_ids = torch.cat([
            current_cat_ids.unsqueeze(1),
            history_cat_ids,
        ], dim=1)
        all_num_values = torch.cat([
            current_num_values.unsqueeze(1),
            history_num_values,
        ], dim=1)
        all_masks = torch.cat([
            current_mask.unsqueeze(1),
            history_mask,
        ], dim=1)

        all_cat_labels = torch.cat([
            current_cat_labels.unsqueeze(1),
            history_cat_labels,
        ], dim=1)
        all_num_labels = torch.cat([
            current_num_labels.unsqueeze(1),
            history_num_labels,
        ], dim=1)

        flat_cat_ids = all_cat_ids.reshape(batch_size * total_transactions, -1)
        flat_num_values = all_num_values.reshape(batch_size * total_transactions, -1)
        flat_masks = all_masks.reshape(batch_size * total_transactions, -1)

        flat_field_embeddings = self.encode_transaction(
            flat_cat_ids,
            flat_num_values,
            flat_masks,
        )

        all_field_embeddings = flat_field_embeddings.reshape(
            batch_size,
            total_transactions,
            num_cat_fields,
            self.field_vector_size,
        )

        sequence_output = all_field_embeddings.reshape(
            batch_size,
            total_transactions * num_cat_fields,
            self.field_vector_size,
        )
        sequence_mask = all_masks.reshape(batch_size, total_transactions * num_cat_fields)

        outputs = self.history_bert(
            inputs_embeds=sequence_output,
            attention_mask=sequence_mask,
        )

        field_embeddings = outputs.last_hidden_state.reshape(
            batch_size,
            total_transactions,
            num_cat_fields,
            self.field_vector_size,
        )

        prediction_scores = self.mlm_head(field_embeddings)
        pooled_transaction_embeddings = self._pool_transaction_fields(field_embeddings, all_masks)
        # num_scores = self.num_head(pooled_transaction_embeddings)

        return prediction_scores, all_cat_labels

    def forward(
        self,
        cat_ids,
        num_values,
        attention_mask,
        labels=None,
    ):
        sequence_output = self._encode_cls_sequence(cat_ids, num_values, attention_mask)
        batch_size, seq_len, _ = sequence_output.shape

        transaction_mask = attention_mask[:, :, 0]
        normalized_num_values = self._normalize_num_values(num_values, transaction_mask)
        classifier_input = torch.cat([sequence_output, normalized_num_values], dim=-1)
        logits = self.classifier(classifier_input.reshape(-1, self.classifier_input_size))
        logits = logits.reshape(batch_size, seq_len, -1)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            active_loss = (labels != -100).view(-1)
            active_logits = logits.view(-1, logits.size(-1))[active_loss]
            active_labels = labels.view(-1)[active_loss]
            loss = loss_fn(active_logits, active_labels)

        return loss, logits

In [10]:
field_ranges = {'user': {'min': 5, 'max': 2004, 'size': 2000},
 'Card': {'min': 2005, 'max': 2013, 'size': 9},
 'Month': {'min': 2014, 'max': 2025, 'size': 12},
 'Day': {'min': 2026, 'max': 2056, 'size': 31},
 'hour': {'min': 2057, 'max': 2080, 'size': 24},
 'minute': {'min': 2081, 'max': 2140, 'size': 60},
 'Use Chip': {'min': 2141, 'max': 2143, 'size': 3},
 'Merchant Name': {'min': 2144, 'max': 102486, 'size': 100343},
 'Merchant City': {'min': 102487, 'max': 115915, 'size': 13429},
 'Merchant State': {'min': 115916, 'max': 116139, 'size': 224},
 'Zip': {'min': 116140, 'max': 143461, 'size': 27322},
 'MCC': {'min': 143462, 'max': 143570, 'size': 109},
 'Errors?': {'min': 143571, 'max': 143594, 'size': 24},
 'isrefund': {'min': 143595, 'max': 143596, 'size': 2}}

inv_vocab = {v: k for k, v in vocab_builder.vocab.items()}

In [ ]:
mlm_model = TabBertForFraudDetectionNumericConcat(
    len(vocab_builder.vocab),
    field_ranges=field_ranges,
    inv_vocab=inv_vocab,
    hidden_size=128
)


# Загрузка параметров модели и vocab
save_path = 'tabbert_numeric_concat/tabbert_mlm_categorical_only_epoch_4.pth'
checkpoint = torch.load(save_path, map_location='cpu', weights_only=False)
mlm_model.load_state_dict(checkpoint['model_state_dict'])

# Загрузка checkpoint


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



mlm_model = mlm_model.to(device)  # Явный перенос на CUDA


Using device: cuda


In [12]:
print("Creating MLM dataset...")

full_dataset = TabBertMLMDataset(
    df, tokenizer, categorical_columns, numerical_columns,  max_seq_length=18
)

Creating MLM dataset...
Создано 2437590 окон из 24386900 транзакций


In [15]:
# del mlm_loader, mlm_val_loader

In [13]:
import os

# Разделяем на train и val (например, 80/20)
from torch.utils.data import random_split

torch.manual_seed(42)

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Total samples: {len(full_dataset)}")
print(f"Train samples: {len(train_dataset)} ({len(train_dataset)/len(full_dataset)*100:.1f}%)")
print(f"Val samples: {len(val_dataset)} ({len(val_dataset)/len(full_dataset)*100:.1f}%)")

# Для машины с 10 CPU cores обычно разумно оставить 2 ядра системе и остальное отдать DataLoader.
AVAILABLE_CPU = os.cpu_count() or 2
NUM_WORKERS = min(8, max(2, AVAILABLE_CPU - 2))
BATCH_SIZE = 64
PIN_MEMORY = device.type == 'cuda'

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
    'persistent_workers': NUM_WORKERS > 0,
    'drop_last': False
}

if NUM_WORKERS > 0:
    loader_kwargs['prefetch_factor'] = 4

mlm_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs
)

mlm_val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

print(f"\nCPU cores available: {AVAILABLE_CPU}")
print(f"Using DataLoader workers: {NUM_WORKERS}")
print(f"Train batches: {len(mlm_loader)}")
print(f"Val batches: {len(mlm_val_loader)}")

Total samples: 2437590
Train samples: 1950072 (80.0%)
Val samples: 487518 (20.0%)

CPU cores available: 44
Using DataLoader workers: 8
Train batches: 30470
Val batches: 7618


In [14]:
from tqdm.auto import tqdm

def train_mlm_categorical_only(
    model,
    train_loader,
    val_loader=None,
    epochs=3,
    lr=2e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    save_dir="tabbert_numeric_concat",
    device=None,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_steps = 0

        pbar = tqdm(train_loader, desc=f"MLM train epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                prediction_scores, cat_labels = model.forward_mlm(
                    current_cat_ids=batch["current_cat_ids"],
                    current_num_values=batch["current_num_values"],
                    current_mask=batch["current_mask"],
                    history_cat_ids=batch["history_cat_ids"],
                    history_num_values=batch["history_num_values"],
                    history_mask=batch["history_mask"],
                    current_cat_labels=batch["current_cat_labels"],
                    current_num_labels=batch["current_num_labels"],
                    history_cat_labels=batch["history_cat_labels"],
                    history_num_labels=batch["history_num_labels"],
                )

                active = cat_labels != -100
                if not active.any():
                    continue

                loss = loss_fn(
                    prediction_scores[active],
                    cat_labels[active],
                )

            scaler.scale(loss).backward()

            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            total_steps += 1
            pbar.set_postfix(loss=total_loss / max(total_steps, 1))

        train_loss = total_loss / max(total_steps, 1)

        val_loss = None
        if val_loader is not None:
            model.eval()
            val_total = 0.0
            val_steps = 0

            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"MLM val epoch {epoch}"):
                    batch = {k: v.to(device) for k, v in batch.items()}

                    prediction_scores, cat_labels = model.forward_mlm(
                        current_cat_ids=batch["current_cat_ids"],
                        current_num_values=batch["current_num_values"],
                        current_mask=batch["current_mask"],
                        history_cat_ids=batch["history_cat_ids"],
                        history_num_values=batch["history_num_values"],
                        history_mask=batch["history_mask"],
                        current_cat_labels=batch["current_cat_labels"],
                        current_num_labels=batch["current_num_labels"],
                        history_cat_labels=batch["history_cat_labels"],
                        history_num_labels=batch["history_num_labels"],
                    )

                    active = cat_labels != -100
                    if not active.any():
                        continue

                    loss = loss_fn(prediction_scores[active], cat_labels[active])
                    val_total += loss.item()
                    val_steps += 1

            val_loss = val_total / max(val_steps, 1)

        print(f"epoch={epoch} train_loss={train_loss:.4f} val_loss={val_loss}")

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            f"{save_dir}/tabbert_mlm_categorical_only_epoch_continue_{epoch}.pth",
        )

    return model

In [ ]:
mlm_model = train_mlm_categorical_only(
    model=mlm_model,
    train_loader=mlm_loader,
    val_loader=mlm_val_loader,
    epochs=3,
    lr=2e-5,
    save_dir="tabbert_numeric_concat",
)

/tmp/job-3953593/ipykernel_248003/1081656013.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


MLM train epoch 1:   0%|          | 0/30470 [00:01<?, ?it/s]

/tmp/job-3953593/ipykernel_248003/1081656013.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


MLM val epoch 1:   0%|          | 0/7618 [00:01<?, ?it/s]

In [26]:
import pandas as pd

def split_by_time_per_user(
    df,
    user_col="user",
    time_col="timestamp",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

    df = df.sort_values([user_col, time_col]).reset_index(drop=True)

    train_parts = []
    val_parts = []
    test_parts = []

    for _, group in df.groupby(user_col, sort=False):
        n = len(group)

        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)

        # чтобы не развалиться на маленьких группах
        if train_end == 0:
            train_end = min(1, n)
        if val_end <= train_end and n - train_end > 1:
            val_end = train_end + 1

        train_parts.append(group.iloc[:train_end])
        val_parts.append(group.iloc[train_end:val_end])
        test_parts.append(group.iloc[val_end:])

    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)

    return train_df, val_df, test_df

In [27]:
train_df, val_df, test_df = split_by_time_per_user(
    df,
    user_col="user",
    time_col="timestamp",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

print(len(train_df), len(val_df), len(test_df))
print(len(train_df) / len(df), len(val_df) / len(df), len(test_df) / len(df))

17069880 3657092 3659928
0.6999610446592228 0.14996133169857587 0.15007762364220134


In [ ]:
train_dataset = TabBertFraudDataset(
    train_df, tokenizer, categorical_columns, numerical_columns, 
    max_seq_length=18, window_size=10,
)

val_dataset = TabBertFraudDataset(
    val_df, tokenizer, categorical_columns, numerical_columns, 
    max_seq_length=18, window_size=10,
)

test_dataset = TabBertFraudDataset(
    test_df, tokenizer, categorical_columns, numerical_columns, 
    max_seq_length=18,  window_size=10,
)

Создано 1707675 fraud-окон из 17069880 транзакций
Создано 366414 fraud-окон из 3657092 транзакций
Создано 366701 fraud-окон из 3659928 транзакций


In [47]:
import os

from torch.utils.data import random_split

torch.manual_seed(42)


print(f"Train samples: {len(train_dataset)} ")
print(f"Val samples: {len(val_dataset)} ")
print(f"Test samples: {len(test_dataset)}")

AVAILABLE_CPU = os.cpu_count() or 2
NUM_WORKERS = min(8, max(2, AVAILABLE_CPU - 2))
BATCH_SIZE = 128
PIN_MEMORY = device.type == 'cuda'

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
    'persistent_workers': NUM_WORKERS > 0,
    'drop_last': False
}

if NUM_WORKERS > 0:
    loader_kwargs['prefetch_factor'] = 4

fraud_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs
)

fraud_val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

fraud_test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

print(f"\nCPU cores available: {AVAILABLE_CPU}")
print(f"Using DataLoader workers: {NUM_WORKERS}")


Train samples: 1707675 
Val samples: 366414 
Test samples: 366701

CPU cores available: 44
Using DataLoader workers: 8


In [42]:
# del fraud_val_loader, fraud_test_loader, mlm_model

In [48]:
import os
import torch
from copy import deepcopy
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np


def _safe_binary_auc(y_true, y_score):
    """Безопасный расчет AUC: возвращает NaN, если в выборке только один класс."""
    unique_classes = np.unique(y_true)
    if unique_classes.shape[0] < 2:
        return float('nan')
    return roc_auc_score(y_true, y_score)


def train_tabbert(
    model,
    train_loader,
    val_loader,
    epochs=5,
    device='cuda',
    save_dir='tabbert_numeric_concat',
    save_prefix='tabbert_fraud_v2'
):
    """Fine-tuning для fraud detection в полном FP32 с early stopping и scheduler по val F1."""
    model = model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    optimizer = torch.optim.AdamW([
        {'params': model.cat_embedding.parameters(), 'lr': 1e-5},
        {'params': model.transaction_bert.parameters(), 'lr': 1e-5},
        {'params': model.history_bert.parameters(), 'lr': 2e-5},
        {'params': model.num_norm.parameters(), 'lr': 2e-5},
        {'params': model.classifier.parameters(), 'lr': 2e-5},
    ], weight_decay=0.1)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2
    )

    best_val_f1 = 0.0
    patience = 5
    patience_counter = 0
    best_model_state = None

    history = {
        'train_loss': [], 'val_loss': [],
        'train_precision': [], 'val_precision': [],
        'train_recall': [], 'val_recall': [],
        'train_f1': [], 'val_f1': [],
        'train_auc': [], 'val_auc': []
    }

    for epoch in range(epochs):
        print(f"\n{'=' * 50}")
        print(f"Эпоха {epoch + 3}/{epochs}")
        print('=' * 50)

        model.train()
        train_loss = 0.0
        train_preds = []
        train_probs = []
        train_labels = []

        train_bar = tqdm(train_loader, desc='Training')
        for batch in train_bar:
            cat_ids = batch['cat_ids'].to(device, non_blocking=True)
            num_values = batch['num_values'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            loss, logits = model(
                cat_ids=cat_ids,
                num_values=num_values,
                attention_mask=attention_mask,
                labels=labels
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)
            preds = logits.argmax(dim=-1)
            valid_mask = (labels != -100)

            valid_preds = preds[valid_mask].detach().cpu().numpy()
            valid_probs = probs[..., 1][valid_mask].detach().cpu().numpy()
            valid_labels = labels[valid_mask].detach().cpu().numpy()

            train_preds.extend(valid_preds.tolist())
            train_probs.extend(valid_probs.tolist())
            train_labels.extend(valid_labels.tolist())

            train_bar.set_postfix(loss=f'{loss.item():.4f}')

        train_loss = train_loss / len(train_loader)
        train_precision = precision_score(train_labels, train_preds, zero_division=0)
        train_recall = recall_score(train_labels, train_preds, zero_division=0)
        train_f1 = f1_score(train_labels, train_preds, zero_division=0)
        train_auc = _safe_binary_auc(np.array(train_labels), np.array(train_probs))

        model.eval()
        val_loss = 0.0
        val_preds = []
        val_probs = []
        val_labels = []

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc='Validation')
            for batch in val_bar:
                cat_ids = batch['cat_ids'].to(device, non_blocking=True)
                num_values = batch['num_values'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)

                loss, logits = model(
                    cat_ids=cat_ids,
                    num_values=num_values,
                    attention_mask=attention_mask,
                    labels=labels
                )

                val_loss += loss.item()

                probs = torch.softmax(logits, dim=-1)
                preds = logits.argmax(dim=-1)
                valid_mask = (labels != -100)

                valid_preds = preds[valid_mask].detach().cpu().numpy()
                valid_probs = probs[..., 1][valid_mask].detach().cpu().numpy()
                valid_labels_batch = labels[valid_mask].detach().cpu().numpy()

                val_preds.extend(valid_preds.tolist())
                val_probs.extend(valid_probs.tolist())
                val_labels.extend(valid_labels_batch.tolist())

                val_bar.set_postfix(loss=f'{loss.item():.4f}')

        val_loss = val_loss / len(val_loader)
        val_precision = precision_score(val_labels, val_preds, zero_division=0)
        val_recall = recall_score(val_labels, val_preds, zero_division=0)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        val_auc = _safe_binary_auc(np.array(val_labels), np.array(val_probs))

        scheduler.step(val_f1)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_precision'].append(train_precision)
        history['val_precision'].append(val_precision)
        history['train_recall'].append(train_recall)
        history['val_recall'].append(val_recall)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        history['train_auc'].append(train_auc)
        history['val_auc'].append(val_auc)

        print(f"\n📊 РЕЗУЛЬТАТЫ ЭПОХИ {epoch + 3}:")
        print(f"  Train - Loss: {train_loss:.4f}")
        print(f"          Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
        print(f"          AUC: {train_auc:.4f}" if not np.isnan(train_auc) else "          AUC: nan")
        print(f"  Val   - Loss: {val_loss:.4f}")
        print(f"          Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
        print(f"          AUC: {val_auc:.4f}" if not np.isnan(val_auc) else "          AUC: nan")

        val_probs_array = np.array(val_probs)
        val_labels_array = np.array(val_labels)
        print(f"\n📈 Статистика вероятностей (val):")
        print(f"  Всего: min={val_probs_array.min():.4f}, max={val_probs_array.max():.4f}, mean={val_probs_array.mean():.4f}")

        fraud_mask = val_labels_array == 1
        if fraud_mask.sum() > 0:
            print(f"  Фрод (класс 1): mean={val_probs_array[fraud_mask].mean():.4f}, max={val_probs_array[fraud_mask].max():.4f}")

        non_fraud_mask = val_labels_array == 0
        if non_fraud_mask.sum() > 0:
            print(f"  Не-фрод (класс 0): mean={val_probs_array[non_fraud_mask].mean():.4f}, max={val_probs_array[non_fraud_mask].max():.4f}")
        print('-' * 50)

        checkpoint_path = os.path.join(save_dir, f"{save_prefix}_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': best_val_f1,
            'history': history,
            'val_metrics': {
                'loss': val_loss,
                'precision': val_precision,
                'recall': val_recall,
                'f1': val_f1,
                'auc': val_auc,
            },
            'train_metrics': {
                'loss': train_loss,
                'precision': train_precision,
                'recall': train_recall,
                'f1': train_f1,
                'auc': train_auc,
            }
        }, checkpoint_path)
        print(f"💾 Сохранен checkpoint: {checkpoint_path}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = deepcopy(model.state_dict())
            patience_counter = 0
            print(f"✅ НОВЫЙ ЛУЧШИЙ F1: {best_val_f1:.4f}")

            best_model_path = os.path.join(save_dir, f"{save_prefix}_best.pth")
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_f1': best_val_f1,
                'history': history,
                'val_metrics': {
                    'loss': val_loss,
                    'precision': val_precision,
                    'recall': val_recall,
                    'f1': val_f1,
                    'auc': val_auc,
                }
            }, best_model_path)
            print(f"🏆 Сохранена лучшая модель: {best_model_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Ранняя остановка на эпохе {epoch + 1}")
                # break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n🏆 Загружена лучшая модель с F1 = {best_val_f1:.4f}")

    return model, history


def plot_training_history(history):
    """Построение графиков обучения."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0, 0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    axes[0, 1].plot(epochs, history['train_f1'], 'b-o', label='Train F1', linewidth=2)
    axes[0, 1].plot(epochs, history['val_f1'], 'r-o', label='Val F1', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    axes[0, 2].plot(epochs, history['train_auc'], 'b-o', label='Train AUC', linewidth=2)
    axes[0, 2].plot(epochs, history['val_auc'], 'r-o', label='Val AUC', linewidth=2)
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('AUC')
    axes[0, 2].set_title('AUC-ROC')
    axes[0, 2].legend()
    axes[0, 2].grid(True)

    axes[1, 0].plot(epochs, history['train_precision'], 'b-o', label='Train Precision', linewidth=2)
    axes[1, 0].plot(epochs, history['val_precision'], 'r-o', label='Val Precision', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    axes[1, 1].plot(epochs, history['train_recall'], 'b-o', label='Train Recall', linewidth=2)
    axes[1, 1].plot(epochs, history['val_recall'], 'r-o', label='Val Recall', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)


In [ ]:
mlm_model = TabBertForFraudDetectionNumericConcat(
    len(vocab_builder.vocab),
    field_ranges=field_ranges,
    inv_vocab=inv_vocab,
    hidden_size=128
)


# Загрузка параметров модели и vocab
save_path = 'tabbert_numeric_concat/tabbert_mlm_categorical_only_epoch_3.pth'
checkpoint = torch.load(save_path, map_location='cpu', weights_only=False)
mlm_model.load_state_dict(checkpoint['model_state_dict'])

# Загрузка checkpoint


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



mlm_model = mlm_model.to(device)  # Явный перенос на CUDA


In [ ]:
new_model, history = train_tabbert(
    mlm_model,
    fraud_train_loader,
    fraud_val_loader,
    epochs=20,)


Эпоха 3/20


Validation: 100%|██████████| 2863/2863 [00:55<00:00, 51.91it/s, loss=0.0005]



📊 РЕЗУЛЬТАТЫ ЭПОХИ 3:
  Train - Loss: 0.0274
          Precision: 0.4496, Recall: 0.2485, F1: 0.3201
          AUC: 0.9031
  Val   - Loss: 0.0093
          Precision: 0.6174, Recall: 0.7430, F1: 0.6744
          AUC: 0.9855

📈 Статистика вероятностей (val):
  Всего: min=0.0000, max=0.9866, mean=0.0022
  Фрод (класс 1): mean=0.7132, max=0.9866
  Не-фрод (класс 0): mean=0.0012, max=0.9733
--------------------------------------------------
💾 Сохранен checkpoint: tabbert_numeric_concat/tabbert_fraud_v2_epoch_0.pth
✅ НОВЫЙ ЛУЧШИЙ F1: 0.6744
🏆 Сохранена лучшая модель: tabbert_numeric_concat/tabbert_fraud_v2_best.pth

Эпоха 4/20


Validation: 100%|██████████| 2863/2863 [00:51<00:00, 55.91it/s, loss=0.0007]



📊 РЕЗУЛЬТАТЫ ЭПОХИ 4:
  Train - Loss: 0.0103
          Precision: 0.7373, Recall: 0.7106, F1: 0.7237
          AUC: 0.9567
  Val   - Loss: 0.0060
          Precision: 0.6723, Recall: 0.8584, F1: 0.7541
          AUC: 0.9918

📈 Статистика вероятностей (val):
  Всего: min=0.0000, max=0.9995, mean=0.0024
  Фрод (класс 1): mean=0.8405, max=0.9995
  Не-фрод (класс 0): mean=0.0013, max=0.9966
--------------------------------------------------
💾 Сохранен checkpoint: tabbert_numeric_concat/tabbert_fraud_v2_epoch_1.pth
✅ НОВЫЙ ЛУЧШИЙ F1: 0.7541
🏆 Сохранена лучшая модель: tabbert_numeric_concat/tabbert_fraud_v2_best.pth

Эпоха 5/20


Training:  91%|█████████ | 12109/13342 [10:39<01:01, 20.20it/s, loss=0.0020]

In [ ]:
mlm_model = TabBertForFraudDetectionNumericConcat(
    len(vocab_builder.vocab),
    field_ranges=field_ranges,
    inv_vocab=inv_vocab,
    hidden_size=128
)


# Загрузка параметров модели и vocab
save_path = 'tabbert_numeric_concat/tabbert_fraud_v2_epoch_15.pth'
checkpoint = torch.load(save_path, map_location='cpu', weights_only=False)
mlm_model.load_state_dict(checkpoint['model_state_dict'])

# Загрузка checkpoint


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



mlm_model = mlm_model.to(device)  # Явный перенос на CUDA


Using device: cuda


In [55]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


def safe_binary_auc(y_true, y_score):
    unique_classes = np.unique(y_true)
    if unique_classes.shape[0] < 2:
        return float('nan')
    return roc_auc_score(y_true, y_score)


def collect_model_probs(model, loader, device='cuda'):
    model.eval()
    model = model.to(device)

    use_amp_local = str(device).startswith('cuda') and torch.cuda.is_available()

    all_probs = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Collecting probs"):
            cat_ids = batch['cat_ids'].to(device, non_blocking=True)
            num_values = batch['num_values'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp_local):
                loss, logits = model(
                    cat_ids=cat_ids,
                    num_values=num_values,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)[..., 1]
            valid_mask = (labels != -100)

            batch_probs = probs[valid_mask].detach().cpu().numpy()
            batch_labels = labels[valid_mask].detach().cpu().numpy()

            all_probs.extend(batch_probs.tolist())
            all_labels.extend(batch_labels.tolist())

    avg_loss = total_loss / len(loader)

    return np.array(all_probs), np.array(all_labels), avg_loss


def compute_metrics_at_threshold(y_true, y_prob, threshold=0.5, loss=None):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        'threshold': threshold,
        'loss': loss,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': safe_binary_auc(y_true, y_prob)
    }


def find_best_threshold(y_true, y_prob, loss=None, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    best_metrics = None
    best_f1 = -1.0

    for thr in thresholds:
        metrics = compute_metrics_at_threshold(y_true, y_prob, threshold=thr, loss=loss)
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            best_metrics = metrics

    return best_metrics


# 1. Собираем вероятности на validation
val_probs, val_labels, val_loss = collect_model_probs(mlm_model, fraud_val_loader, device=device)

# 2. Ищем лучший threshold по val F1
best_val_metrics = find_best_threshold(val_labels, val_probs, loss=val_loss)
best_threshold = best_val_metrics['threshold']

print(f"Best threshold on val: {best_threshold:.2f}")
print("Validation metrics at best threshold:")
print(best_val_metrics)

# 3. Считаем test метрики с этим threshold
test_probs, test_labels, test_loss = collect_model_probs(mlm_model, fraud_test_loader, device=device)
test_metrics = compute_metrics_at_threshold(test_labels, test_probs, threshold=best_threshold, loss=test_loss)

print("\nTest metrics with validation-selected threshold:")
print(test_metrics)

Best threshold on val: 0.89
Validation metrics at best threshold:
{'threshold': np.float64(0.8900000000000002), 'loss': 0.0044967467989591265, 'precision': 0.9395617070357555, 'recall': 0.8203423967774421, 'f1': 0.8759139784946236, 'auc': np.float64(0.9952750875959536)}



Test metrics with validation-selected threshold:
{'threshold': np.float64(0.8900000000000002), 'loss': 0.002561892741571478, 'precision': 0.9449639326743254, 'recall': 0.8622623110677718, 'f1': 0.9017208413001913, 'auc': np.float64(0.9991501642750718)}
